{
    "Name": "Aphasia Recovery Cohort (ARC) Dataset",
    "BIDSVersion": "1.6.0",
    "DatasetType": "raw",
    "Authors": [
        "Makayla Gibson",
        "Roger Newman-Norlund",
        "Leonardo Bonilha",
        "Julius Fridriksson",
        "Gregory Hickok",
        "Argye E. Hillis",
        "Dirk-Bart den Ouden",
        "Chris Rorden"
    ],
    "Acknowledgements": "Special thanks to the C-STAR participants, faculty and staff, as well as our collaborators",
    "Funding": [
        "National Institute of Health Grant P50DC014664",
        "National Institute of Health Grant U01DC011739",
        "National Institute of Health Grant R01DC008355",
        "National Institute of Health Grant RF1MH133701"
    ],
    "EthicsApprovals": [
        "All data acquired from studies that were approved by the Institutional Review Board at the University of South Carolina. Because final dataset is fully anonymised, it meets exemption criteria."
    ],
    "GeneratedBy": [
        {
            "Name": "Manual"
        }
    ],
    "DatasetDOI": "doi:10.18112/openneuro.ds004884.v1.0.2",
    "License": "CC0",
    "HowToAcknowledge": "Gibson M, Newman-Norlund R, Bonilha L, Fridriksson J, Hickok G, Hillis AE, den Ouden DB, Rorden C. The Aphasia Recovery Cohort, an open-source chronic stroke repository. Sci Data. 2024 Sep 9;11(1):981. doi: 10.1038/s41597-024-03819-7. PMID: 39251640.\n\nPMID: 39251640  \n\nDOI: 10.1038/s41597-024-03819-7"
}

"Images from individuals with chronic stroke who have experienced speech and language impairments. Many individuals have participated in multiple sessions, allowing longitudinal studies."

______________________________________

How to cite: 

"Gibson, M., Newman-Norlund, R., Bonilha, L. et al. The Aphasia Recovery Cohort, an open-source chronic stroke repository. Sci Data 11, 981 (2024). "

DOI: 10.1038/s41597-024-03819-7
PMID: 39251640  

In [15]:
# Modality & Lesion Mask Coverage Analysis (ARC chronic stroke)
import pandas as pd, re, unicodedata, json
from pathlib import Path
from collections import Counter, defaultdict
from shutil import copy2

modality_tokens = {'T1w', 'T2w', 'FLAIR', 'PD', 'PDw', 'T2star', 'SWI', 'bold', 'sbref', 'dwi', 'angio', 'T1map', 'T2map', 'CBF', 'CBV'}
modality_context = {
    'T1w': 'High-resolution structural imaging for cortical anatomy and longitudinal alignment.',
    'T2w': 'Sensitive to chronic lesion cavities that inform lesion mask delineation.',
    'FLAIR': 'Suppresses CSF to highlight periventricular and chronic white-matter hyperintensities.',
    'bold': 'Task-based fMRI probing language networks (e.g., naming paradigms).',
    'dwi': 'Diffusion-weighted scans capturing white-matter integrity and tract status.',
    'sbref': 'Single-band reference frames supporting distortion correction for BOLD acquisitions.'
}
modalities_session_counter = Counter()
modalities_patient_map = defaultdict(set)

def _extract_modalities(filename: str) -> set:
    stem = filename[:-7] if filename.endswith('.nii.gz') else filename[:-4] if filename.endswith('.nii') else filename
    segments = stem.split('_')
    mods = {seg for seg in segments if seg in modality_tokens}
    lname = filename.lower()
    if 'bold' in lname:
        mods.add('bold')
    if 'dwi' in lname:
        mods.add('dwi')
    if 'sbref' in lname:
        mods.add('sbref')
    return mods

# Reconstruct base and (re)load participants with classification if prior cell not run
base = Path('/home/rbielski/ARC/ds004884')
participants_path = base / 'participants.tsv'
na_vals = ['n/a', 'NA', 'NaN', '', 'NULL', 'null']

def _clean_col(c: str) -> str:
    c = unicodedata.normalize('NFKC', c).replace('\ufeff','')
    return re.sub(r'\s+', '_', c.strip().lower())

if 'df' not in globals() or 'stroke_status' not in globals().get('df', pd.DataFrame()).columns:
    df = pd.read_csv(participants_path, sep='\t', na_values=na_vals, dtype=str)
    orig_cols = list(df.columns)
    df.columns = [_clean_col(c) for c in df.columns]
    print(f"Original columns: {orig_cols}\nNormalized columns: {list(df.columns)}")

    # Detect participant id column robustly
    pid_col = None
    for c in df.columns:
        if c.startswith('participant'):
            pid_col = c; break
    if pid_col and pid_col != 'participant_id':
        df.rename(columns={pid_col: 'participant_id'}, inplace=True)
        print(f"✓ Renamed '{pid_col}' -> 'participant_id'")

    if 'participant_id' not in df.columns:
        print("⚠️ 'participant_id' column missing after normalization; creating from index.")
        df['participant_id'] = [f'sub-idx{i}' for i in range(len(df))]

    for c in ['acuteischaemicstroke', 'priorstroke']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    def classify(row):
        a = row.get('acuteischaemicstroke'); p = row.get('priorstroke')
        if pd.isna(a) and pd.isna(p): return 'unknown'
        if pd.isna(a) or pd.isna(p): return 'partial_unknown'
        if a == 1 and p == 1: return 'acute+chronic'
        if a == 1 and p == 0: return 'acute_only'
        if a == 0 and p == 1: return 'chronic_only'
        if a == 0 and p == 0: return 'no_stroke_reported'
        return 'unclassified'

    if 'stroke_status' not in df.columns:
        df['stroke_status'] = df.apply(classify, axis=1)

# ---------------------- new defensive guard (patch) ----------------------
# Ensure both required columns exist; if not, force a clean reload
required = {'participant_id','stroke_status'}
missing_now = [c for c in required if c not in df.columns]

if missing_now:
    print(f"♻️ Re-loading participants.tsv because missing columns: {missing_now}")
    df_reload = pd.read_csv(participants_path, sep='\t', na_values=na_vals, dtype=str)
    df_reload.columns = [_clean_col(c) for c in df_reload.columns]

    # Rebuild / rename participant id
    if 'participant_id' not in df_reload.columns:
        candidate = next((c for c in df_reload.columns if c.startswith('participant')), None)
        if candidate and candidate != 'participant_id':
            df_reload.rename(columns={candidate: 'participant_id'}, inplace=True)
    if 'participant_id' not in df_reload.columns:
        df_reload['participant_id'] = [f'sub-idx{i}' for i in range(len(df_reload))]

    # Stroke columns
    for c in ['acuteischaemicstroke','priorstroke']:
        if c in df_reload.columns:
            df_reload[c] = pd.to_numeric(df_reload[c], errors='coerce')

    if 'stroke_status' not in df_reload.columns:
        df_reload['stroke_status'] = df_reload.apply(classify, axis=1)

    # Use reloaded frame only if it now satisfies
    if required.issubset(df_reload.columns):
        df = df_reload
        print("✅ Successfully reconstructed missing columns.")
    else:
        # fallback: synthesize minimal columns
        print("⚠️ Could not fully reconstruct; synthesizing placeholders.")
        if 'participant_id' not in df.columns:
            df['participant_id'] = [f'sub-idx{i}' for i in range(len(df))]
        if 'stroke_status' not in df.columns:
            df['stroke_status'] = 'unknown'

# Final sanity
req_cols = {'participant_id','stroke_status'}
missing_req = req_cols - set(df.columns)
if missing_req:
    print(f"❌ Required columns still missing: {missing_req} (will fallback).")

# ---------------------- existing filesystem scan code (unchanged) ----------------------
subjects_fs = sorted([p.name for p in base.glob('sub-*')
                      if p.is_dir() and re.match(r'sub-\w+', p.name)])

session_rows = []
copy_jobs = []
for sid in subjects_fs:
    subj_dir = base / sid
    session_dirs = sorted([p for p in subj_dir.glob('ses-*') if p.is_dir()])
    if not session_dirs:
        session_dirs = [subj_dir]
    for ses in session_dirs:
        ses_label = ses.name if ses.name.startswith('ses-') else 'ses-NA'
        anat_dir = (ses / 'anat') if ses is not subj_dir else subj_dir / 'anat'
        func_dir = (ses / 'func') if ses is not subj_dir else subj_dir / 'func'
        dwi_dir = (ses / 'dwi') if ses is not subj_dir else subj_dir / 'dwi'
        prefix = f"{sid}_{ses_label}" if ses_label != 'ses-NA' else sid
        session_modalities = set()
        t1_files = list(anat_dir.glob(f"{prefix}_*T1w.nii.gz")) if anat_dir.exists() else []
        has_t1w = bool(t1_files)
        has_t2w = any(anat_dir.glob(f"{prefix}_*T2w.nii.gz")) if anat_dir.exists() else False
        has_flair = any(anat_dir.glob(f"{prefix}_*FLAIR.nii.gz")) if anat_dir.exists() else False
        has_bold = any(func_dir.glob(f"{prefix}_*bold.nii.gz")) if func_dir.exists() else False
        mask_dir = base / 'derivatives' / 'lesion_masks' / sid
        if ses_label != 'ses-NA':
            mask_dir = mask_dir / ses_label / 'anat'
        else:
            mask_dir = mask_dir / 'anat'
        has_lesion_mask = False
        mask_files = []
        if mask_dir.exists():
            for mask_file in mask_dir.glob(f"{prefix}_*desc-lesion*_mask.nii.gz"):
                has_lesion_mask = True
                mask_files.append(mask_file)
                session_modalities.update(_extract_modalities(mask_file.name))
        if t1_files and mask_files:
            copy_jobs.append({
                'subject': sid,
                'session': ses_label if ses_label != 'ses-NA' else '',
                't1_files': [str(p) for p in t1_files],
                'mask_files': [str(p) for p in mask_files]
            })
        for mod in session_modalities:
            modalities_session_counter[mod] += 1
            modalities_patient_map[mod].add(sid)
        session_rows.append({
            'participant_id': sid,
            'session_id': ses_label if ses_label != 'ses-NA' else '',
            'has_T1w': has_t1w,
            'has_T2w': has_t2w,
            'has_FLAIR': has_flair,
            'has_func_bold': has_bold,
            'has_lesion_mask': has_lesion_mask
        })

mod_df = pd.DataFrame(session_rows)
modality_patient_counts = {mod: len(subjects) for mod, subjects in modalities_patient_map.items()}

# ---------------------- patched safe merge ----------------------
if req_cols.issubset(df.columns):
    merged = mod_df.merge(df[['participant_id','stroke_status']], on='participant_id', how='left')
else:
    merged = mod_df.copy()
    merged['stroke_status'] = 'chronic_dataset'

multi_session_counts = merged.groupby('participant_id')['session_id'].nunique()
subjects_multi_session = (multi_session_counts > 1).sum()
total_sessions = len(merged)
lesion_masks = merged['has_lesion_mask'].sum()
t2_sources = merged['has_T2w'].sum()
bold_sessions = merged['has_func_bold'].sum()

print('\n' + '='*60)
print('ARC Chronic Stroke Dataset Overview:')
print('='*60)
print(f"Participants examined: {merged['participant_id'].nunique()}")
print(f"Total imaging sessions: {total_sessions}")
print(f"Subjects with >1 session: {subjects_multi_session}")
print(f"Sessions with lesion masks (T2w-derived): {lesion_masks}")
print(f"Sessions with T2-weighted acquisitions: {t2_sources}")
print(f"Sessions with functional BOLD tasks: {bold_sessions}")
print('\nMRI modality coverage by sessions:')
for mod, count in modalities_session_counter.most_common():
    participant_spread = modality_patient_counts.get(mod, 0)
    share = (count / total_sessions * 100) if total_sessions else 0
    print(f" - {mod}: {count} sessions ({share:.1f}%) across {participant_spread} participants")
    context = modality_context.get(mod)
    if context:
        print(f"   • {context}")
print('\n✅ Merge completed with participant_id')


ARC Chronic Stroke Dataset Overview:
Participants examined: 230
Total imaging sessions: 902
Subjects with >1 session: 141
Sessions with lesion masks (T2w-derived): 228
Sessions with T2-weighted acquisitions: 440
Sessions with functional BOLD tasks: 850

MRI modality coverage by sessions:
 - T2w: 228 sessions (25.3%) across 228 participants
   • Sensitive to chronic lesion cavities that inform lesion mask delineation.

✅ Merge completed with participant_id


In [16]:
if 'copy_jobs' not in globals():
    print('⚠️ copy_jobs not defined; run the modality summary cell first.')
else:
    export_root = base / 'derivatives' / 'aggregates' / 't1w_with_masks'
    export_root.mkdir(parents=True, exist_ok=True)
    copied_files = 0
    skipped_files = 0
    for job in copy_jobs:
        for path_str in job['t1_files'] + job['mask_files']:
            src = Path(path_str)
            dest = export_root / src.name
            if dest.exists():
                skipped_files += 1
                continue
            copy2(src, dest)
            copied_files += 1
    print(f'📁 Copied {copied_files} files into {export_root}')
    if skipped_files:
        print(f'ℹ️ Skipped {skipped_files} existing files.')

📁 Copied 0 files into /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks
ℹ️ Skipped 406 existing files.


In [3]:
ROOT = "/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks"



Below is non registerd - orginal ARC T1w and their masks. 

In [14]:
# ==== Native T1 + Lesion Mask viewer (original volumes, no resample) ====
from pathlib import Path
import re, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output
from functools import lru_cache
from scipy.ndimage import binary_dilation

# ---------------- Manual case list ----------------
RAW_CASES = [
    {
        "label": "sub-M2001_raw",
        "image": Path("/home/rbielski/ARC/ds004884/sub-M2001/ses-1076/anat/sub-M2001_ses-1076_acq-tfl3_run-4_T1w.nii.gz"),
        "mask": Path("/home/rbielski/ARC/ds004884/derivatives/lesion_masks/sub-M2001/ses-1253/anat/sub-M2001_ses-1253_acq-spc3_run-3_T2w_desc-lesion_mask.nii.gz")
    },
    # Add more dict entries here as needed
]

native_pairs = {}
missing = []
for case in RAW_CASES:
    label = case.get("label") or Path(case["image"]).stem
    img_path = Path(case["image"]).expanduser()
    mask_path = Path(case["mask"]).expanduser()
    if img_path.exists() and mask_path.exists():
        native_pairs[label] = {"t1": img_path, "mask": mask_path}
    else:
        missing.append((label, img_path.exists(), mask_path.exists()))

if missing:
    for label, img_ok, mask_ok in missing:
        print(f"⚠️ Missing files for {label}: image={img_ok} mask={mask_ok}")
if not native_pairs:
    raise RuntimeError("No valid RAW_CASES entries found. Check the paths above.")

native_keys = sorted(native_pairs.keys())

# ---------------- Helpers ----------------
def _run_int(name):
    m = re.search(r"run-(\d+)", name)
    return int(m.group(1)) if m else None

@lru_cache(maxsize=256)
def _img(path: str) -> nib.Nifti1Image:
    return nib.load(path)

@lru_cache(maxsize=256)
def _vol(path: str) -> np.ndarray:
    arr = nib.load(path).get_fdata()
    if arr.ndim == 4 and arr.shape[-1] == 1:
        arr = arr[..., 0]
    return arr.astype(np.float32)

def _normalize(img: np.ndarray) -> np.ndarray:
    nz = img[np.isfinite(img)]
    nz = nz[nz > 0]
    if nz.size == 0:
        return np.zeros_like(img, dtype=np.float32)
    p1, p99 = np.percentile(nz, [1, 99])
    img = np.clip(img, p1, p99)
    m, s = nz.mean(), nz.std()
    if s > 0:
        img = (img - m) / s
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-8)

def _edges2d(mask2d):
    m = mask2d.astype(bool)
    return binary_dilation(m) & (~m)

def _zooms3(img: nib.Nifti1Image):
    z = img.header.get_zooms()[:3]
    return tuple(float(v) for v in z)

def _aff_equal(a, b, tol=1e-4):
    return np.allclose(a, b, atol=tol)

# ---------------- UI ----------------
dd_case   = W.Dropdown(options=native_keys, description="Case:", layout=W.Layout(width="100%"))
sl_slice  = W.IntSlider(description="Axial slice:", min=0, max=1, value=0, continuous_update=False, layout=W.Layout(width="60%"))
sl_alpha  = W.FloatSlider(description="Mask α:", min=0.0, max=1.0, step=0.05, value=0.55, layout=W.Layout(width="35%"))
cb_edges  = W.Checkbox(description="Edges only", value=True)
cb_invert = W.Checkbox(description="Invert image", value=False)
status = W.HTML(f"<b>Viewer</b> — cases: {len(native_keys)}")
controls = W.VBox([status, dd_case, W.HBox([sl_slice, sl_alpha]), W.HBox([cb_edges, cb_invert])])
out = W.Output()

def _update_slice_range(*_):
    key = dd_case.value
    t1 = _img(str(native_pairs[key]["t1"]))
    sl_slice.max = max(0, t1.shape[2] - 1)
    sl_slice.value = min(sl_slice.value, sl_slice.max)

def _draw(*_):
    with out:
        clear_output(wait=True)
        key = dd_case.value
        t1_path = native_pairs[key]["t1"]
        mask_path = native_pairs[key]["mask"]

        t1_img   = _img(str(t1_path))
        mask_img = _img(str(mask_path))
        t1_vol   = _vol(str(t1_path))
        mask_vol = _vol(str(mask_path))

        same_shape  = t1_vol.shape[:3] == mask_vol.shape[:3]
        same_affine = _aff_equal(t1_img.affine, mask_img.affine)

        sl_slice.max = max(0, t1_vol.shape[2] - 1)
        idx = int(sl_slice.value)

        img2d = t1_vol[:, :, idx]
        img2d = _normalize(img2d)
        if cb_invert.value:
            img2d = 1.0 - img2d

        fig, axes = (plt.subplots(1, 2, figsize=(10, 5)) if not same_shape else (plt.subplots(1, 1, figsize=(5.6, 5.6))))
        if not same_shape:
            axes = np.atleast_1d(axes)

        if same_shape:
            mask2d = mask_vol[:, :, idx] > 0.5
            plt.imshow(img2d.T, cmap="gray", origin="lower")
            if cb_edges.value:
                plt.contour(_edges2d(mask2d).T, levels=[0.5], linewidths=0.8, colors="r")
            else:
                plt.imshow(np.ma.masked_where(~mask2d.T, mask2d.T), cmap="jet", alpha=float(sl_alpha.value), origin="lower")
            plt.axis("off"); plt.tight_layout(); plt.show(); plt.close()
        else:
            axes[0].imshow(img2d.T, cmap="gray", origin="lower")
            axes[0].set_title("Image slice")
            axes[0].axis("off")
            mask_slice = mask_vol[:, :, min(idx, mask_vol.shape[2]-1)]
            axes[1].imshow(mask_slice.T, cmap="hot", origin="lower")
            axes[1].set_title("Mask slice (native)")
            axes[1].axis("off")
            plt.tight_layout(); plt.show(); plt.close()

        status.value = (
            f"<b>{key}</b> | image: {t1_vol.shape[:3]} {tuple(round(z,3) for z in _zooms3(t1_img))} | "
            f"mask: {mask_vol.shape[:3]} {tuple(round(z,3) for z in _zooms3(mask_img))} | "
            f"affine match: {'✅' if same_affine else '⚠️'} | overlay: {'✅' if same_shape and same_affine else '❌'}"
        )
        print("Image:", t1_path.name)
        print("Mask :", mask_path.name)
        if not same_shape or not same_affine:
            print("⚠️ Shapes or affines differ; mask shown separately with no resampling.")

dd_case.observe(_update_slice_range, names="value")
dd_case.observe(_draw, names="value")
sl_slice.observe(_draw, names="value")
sl_alpha.observe(_draw, names="value")
cb_edges.observe(_draw, names="value")
cb_invert.observe(_draw, names="value")

_update_slice_range()
_draw()
display(controls, out)

Output()

In [12]:
from templateflow.api import get as tf_get
from pathlib import Path
import nibabel as nib

# Option A: full-head 1mm T1w (usually a single file)
mni_files = tf_get("MNI152NLin2009cAsym", resolution=1, suffix="T1w", extension="nii.gz")
MNI_PATH = Path(mni_files[0])  # <-- pick the first (or add logic below)
print("MNI:", MNI_PATH)

img = nib.load(str(MNI_PATH))
print("shape:", img.shape, "zooms:", img.header.get_zooms())


# Quick sanity check: ANTs + Template present + first pair exists
import shutil, os
from pathlib import Path

ANTS_REG   = shutil.which("antsRegistration") or "/home/rbielski/miniconda3/envs/stroke_env/bin/antsRegistration"
ANTS_APPLY = shutil.which("antsApplyTransforms") or "/home/rbielski/miniconda3/envs/stroke_env/bin/antsApplyTransforms"

mask = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/sub-M2012_ses-1158_acq-spc3_run-4_T2w_desc-lesion_mask.nii.gz")
t1   = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/sub-M2012_ses-1158_acq-tfl3p2_run-3_T1w.nii.gz")

print("antsRegistration:", ANTS_REG, "exists:", Path(ANTS_REG).exists())
print("antsApplyTransforms:", ANTS_APPLY, "exists:", Path(ANTS_APPLY).exists())
print("Mask exists:", mask.exists())
print("T1w  exists:", t1.exists())

# TemplateFlow cache path you previously used:
mni = Path("/home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz")
print("MNI template:", mni, "exists:", mni.exists())

# Optional: print ANTs version
import subprocess
try:
    out = subprocess.run([ANTS_REG, "--version"], capture_output=True, text=True)
    print("\nantsRegistration --version:\n", out.stdout or out.stderr)
except Exception as e:
    print("Version check error:", e)


MNI: /home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz
shape: (193, 229, 193) zooms: (1.0, 1.0, 1.0)
antsRegistration: /home/rbielski/miniconda3/envs/stroke_env/bin/antsRegistration exists: True
antsApplyTransforms: /home/rbielski/miniconda3/envs/stroke_env/bin/antsApplyTransforms exists: True
Mask exists: True
T1w  exists: True
MNI template: /home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz exists: True

antsRegistration --version:
 ANTs Version: 2.6.0.dev1-gb775a15
Compiled: Apr 16 2025 14:14:03




In [11]:
# === Batch: (1) Mask->T1 (NN, bin)  (2) T1->MNI SyN  (3) Apply to T1+mask ===
import os, sys, re, csv, site, shutil, subprocess
from pathlib import Path
from datetime import datetime
import nibabel as nib
import numpy as np
from nibabel.processing import resample_from_to

# ---------- PATHS ----------
ROOT    = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks")
OUT_TMP = ROOT / "._ants_work2"               # transforms / intermediates (second run)
OUT_NAT = ROOT / "native_resampled_masks"     # masks resampled to native T1 grid
OUT_MNI = ROOT / "mni_1mm_ants_fixed"         # final MNI outputs (T1 + mask)
for d in (OUT_TMP, OUT_NAT, OUT_MNI): d.mkdir(exist_ok=True, parents=True)

# ---------- SPEED SETTINGS ----------
# Use 2mm template for registration (big speedup), then apply transforms at 1mm for outputs.
USE_2MM_FOR_REG = True
# Threading for ANTs/ITK (pick what your box can handle)
os.environ.setdefault("ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS", "8")

# ---------- TEMPLATE ----------
try:
    from templateflow.api import get as tf_get
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "--user", "templateflow"], check=True)
    sys.path.append(site.getusersitepackages())
    from templateflow.api import get as tf_get

tpl_id = "MNI152NLin2009cAsym"
mni_1mm = tf_get(tpl_id, resolution=1, suffix="T1w", desc="brain", extension="nii.gz")
MNI_1MM = Path(mni_1mm[0] if isinstance(mni_1mm, (list, tuple)) else mni_1mm)
assert MNI_1MM.exists(), f"Missing template: {MNI_1MM}"

if USE_2MM_FOR_REG:
    mni_2mm = tf_get(tpl_id, resolution=2, suffix="T1w", desc="brain", extension="nii.gz")
    MNI_REG = Path(mni_2mm[0] if isinstance(mni_2mm, (list, tuple)) else mni_2mm)
else:
    MNI_REG = MNI_1MM
assert Path(MNI_REG).exists(), f"Missing registration template: {MNI_REG}"

# ---------- ANTs ----------
ANTS_REG   = shutil.which("antsRegistration")    or "/home/rbielski/miniconda3/envs/stroke_env/bin/antsRegistration"
ANTS_APPLY = shutil.which("antsApplyTransforms") or "/home/rbielski/miniconda3/envs/stroke_env/bin/antsApplyTransforms"
assert Path(ANTS_REG).exists(),   f"antsRegistration not found: {ANTS_REG}"
assert Path(ANTS_APPLY).exists(), f"antsApplyTransforms not found: {ANTS_APPLY}"

def run(cmd, check=True):
    # Add --float 1 for speed/memory where applicable
    if cmd[0].endswith("antsRegistration") and "--float" not in cmd:
        cmd += ["--float", "1"]
    if cmd[0].endswith("antsApplyTransforms") and "--float" not in cmd:
        cmd += ["--float", "1"]
    print(">>", " ".join(cmd))
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if check and res.returncode != 0:
        print(res.stdout)
        raise RuntimeError("Command failed")
    return res.stdout

# ---------- HELPERS ----------
def _tag(s, tag):
    m = re.search(fr"({tag}-[^_]+)", s)
    return m.group(1) if m else None

def key_from_path(p: Path) -> str:
    sub = _tag(p.name, "sub")
    ses = _tag(p.name, "ses")
    return "_".join([x for x in (sub, ses) if x])

def _t1_pref(name: str) -> int:
    n = name.lower()
    if "tfl" in n: return 0
    if "mprage" in n: return 1
    if "mp2rage" in n: return 2
    return 3

def choose_t1_for(key: str) -> Path | None:
    cands = sorted(ROOT.glob(f"{key}*_T1w.nii.gz"))
    if not cands: return None
    cands.sort(key=lambda p: (_t1_pref(p.name), p.name))
    return cands[0]

def resample_mask_to_t1(mask_path: Path, t1_path: Path, out_path: Path):
    mi = nib.load(str(mask_path))
    ti = nib.load(str(t1_path))
    if mi.shape[:3] != ti.shape[:3] or not np.allclose(mi.affine, ti.affine, atol=1e-4):
        rs = resample_from_to(mi, (ti.shape, ti.affine), order=0)   # NN resample
        data = (rs.get_fdata() > 0.5).astype(np.uint8)
    else:
        data = (mi.get_fdata() > 0.5).astype(np.uint8)
    nib.save(nib.Nifti1Image(data, ti.affine, ti.header), str(out_path))

def ants_syn_t1_to_mni(t1_path: Path, prefix: Path):
    # One SyN per key; register to MNI_REG (2mm if enabled)
    run([
        ANTS_REG, "-d","3",
        "-r", f"[{MNI_REG},{t1_path},1]",
        "-m", f"Mattes[{MNI_REG},{t1_path},1,32,Regular,0.25]",
        "-t","Rigid[0.1]","-c","1000x500x250","-s","3x2x1vox","-f","4x2x1",
        "-m", f"Mattes[{MNI_REG},{t1_path},1,32,Regular,0.25]",
        "-t","Affine[0.1]","-c","1000x500x250","-s","3x2x1vox","-f","4x2x1",
        "-m", f"CC[{MNI_REG},{t1_path},1,4]",
        "-t","SyN[0.1,3,0]","-c","60x40x20","-s","2x1x0vox","-f","4x2x1",  # slightly faster schedule
        "-o", f"[{prefix},{prefix}warped.nii.gz,{prefix}invwarped.nii.gz]"
    ])

def apply_to(img_in: Path, ref: Path, xfm_prefix: Path, out_path: Path, nn=False):
    args = [ANTS_APPLY, "-d","3", "-i", str(img_in), "-r", str(ref), "-o", str(out_path)]
    if nn: args += ["-n","NearestNeighbor"]
    args += ["-t", str(xfm_prefix) + "1Warp.nii.gz", "-t", str(xfm_prefix) + "0GenericAffine.mat"]
    run(args)

# ---------- DISCOVER (unique masks) ----------
raw_masks = sorted(ROOT.glob("*_desc-lesion_mask.nii.gz"))
assert raw_masks, "No native lesion masks found."
# De-dup by full filename; keeps stability if the folder has dupes/symlinks
seen = {}
for p in raw_masks:
    seen[p.name] = p
native_masks = list(seen.values())
print(f"[info] masks found: {len(raw_masks)} | unique: {len(native_masks)}")

# ---------- MAIN LOOP (idempotent) ----------
qc_rows = []
ts = datetime.now().isoformat(timespec="seconds")

for m in native_masks:
    key = key_from_path(m)
    t1 = choose_t1_for(key)
    if not t1:
        print(f"[skip] No T1 found for {m.name}")
        continue

    out_mask_t1 = OUT_NAT / f"{key}_lesion_mask_T1w_native.nii.gz"
    xfm_prefix  = OUT_TMP / f"{key}_t1_to_mni_"
    warp_file   = OUT_TMP / f"{key}_t1_to_mni_1Warp.nii.gz"
    aff_file    = OUT_TMP / f"{key}_t1_to_mni_0GenericAffine.mat"
    out_t1_mni  = OUT_MNI / f"{key}_T1w_MNI.nii.gz"
    out_msk_mni = OUT_MNI / f"{key}_lesion_mask_MNI.nii.gz"

    # (1) Mask→T1 grid (NN + bin)
    if not out_mask_t1.exists():
        print(f"\n=== {key} :: resample mask→T1 ===")
        print("T1 :", t1.name)
        print("MSK:", m.name)
        resample_mask_to_t1(m, t1, out_mask_t1)

    # (2) T1→MNI SyN (once per key)
    if not (warp_file.exists() and aff_file.exists()):
        print(f"=== {key} :: antsRegistration (T1→MNI) ===")
        ants_syn_t1_to_mni(t1, xfm_prefix)

    # (3) Apply transforms (T1: linear interp default; Mask: NN)
    if not out_t1_mni.exists():
        apply_to(t1, MNI_1MM, xfm_prefix, out_t1_mni, nn=False)
    if not out_msk_mni.exists():
        apply_to(out_mask_t1, MNI_1MM, xfm_prefix, out_msk_mni, nn=True)
        # hard re-binarize (paranoia)
        mi = nib.load(str(out_msk_mni))
        data = (mi.get_fdata() > 0.5).astype(np.uint8)
        nib.save(nib.Nifti1Image(data, mi.affine, mi.header), str(out_msk_mni))

    # QC row
    ti = nib.load(str(out_t1_mni)); mi = nib.load(str(out_msk_mni))
    qc_rows.append(dict(
        key=key,
        t1_src=str(t1), mask_src=str(m),
        mask_native=str(out_mask_t1),
        t1_mni=str(out_t1_mni), mask_mni=str(out_msk_mni),
        t1_shape=str(ti.shape), t1_zooms=str(tuple(round(z,3) for z in ti.header.get_zooms()[:3])),
        mask_shape=str(mi.shape), mask_zooms=str(tuple(round(z,3) for z in mi.header.get_zooms()[:3])),
        mask_nonzero=int(np.count_nonzero(mi.get_fdata() > 0)),
        timestamp=ts
    ))

# ---------- WRITE QC ----------
qc_csv = OUT_MNI / "qc_summary.csv"
if qc_rows:
    with open(qc_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(qc_rows[0].keys()))
        w.writeheader(); w.writerows(qc_rows)
    print("\nQC written:", qc_csv)
else:
    print("\nNo QC rows written.")


[info] masks found: 203 | unique: 203


[info] masks found: 203 | unique: 203


KeyboardInterrupt: 

In [24]:
# ==== MNI audit + T1w/MNI viewer (auto-detect output dirs) ====
from pathlib import Path
import re, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output
from functools import lru_cache
from scipy.ndimage import binary_dilation



# BASE (the directory that directly contains mni_1mm_ants/ and/or mni_1mm_ants_fixed/)
ROOT = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks")
CANDIDATE_MNI_DIRS = [ROOT / "mni_1mm_ants_fixed", ROOT / "mni_1mm_ants"]
OUT_MNI = next((p for p in CANDIDATE_MNI_DIRS if p.exists()), None)
print("ROOT:", ROOT)
print("Found OUT_MNI:", OUT_MNI)


OUT_NATMSK = ROOT / "native_resampled_masks"  # optional
HAVE_NATIVE = OUT_NATMSK.exists()

TPL = Path("/home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz")
tpl_img = nib.load(str(TPL))
tpl_shape = tpl_img.shape[:3]
tpl_zooms = tuple(float(z) for z in tpl_img.header.get_zooms()[:3])
tpl_aff   = tpl_img.affine

def _key_from_name(p: Path) -> str:
    m_sub = re.search(r"(sub-[^_]+)", p.name)
    m_ses = re.search(r"(ses-[^_]+)", p.name)
    parts = [m_sub.group(1) if m_sub else None, m_ses.group(1) if m_ses else None]
    return "_".join([x for x in parts if x])

@lru_cache(maxsize=256)
def _load_img(path: str) -> nib.Nifti1Image: return nib.load(path)

@lru_cache(maxsize=256)
def _load_vol(path: str) -> np.ndarray:
    arr = nib.load(path).get_fdata()
    if arr.ndim == 4 and arr.shape[-1] == 1: arr = arr[..., 0]
    return arr.astype(np.float32)

def _normalize(img: np.ndarray) -> np.ndarray:
    nz = img[img > 0]
    if nz.size == 0: return np.zeros_like(img, dtype=np.float32)
    p1, p99 = np.percentile(nz, [1, 99]); img = np.clip(img, p1, p99)
    m, s = nz.mean(), nz.std(); img = (img - m) / (s + 1e-8)
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-8)

def _edges2d(m2d): 
    from scipy.ndimage import binary_dilation
    m = m2d.astype(bool); return binary_dilation(m) & (~m)

def _zooms3(img: nib.Nifti1Image):
    z = img.header.get_zooms()[:3]
    return tuple(float(v) for v in z)

def _affine_equal(a: np.ndarray, b: np.ndarray, tol=1e-4): 
    return np.allclose(a, b, atol=tol)

# ---- Collect MNI pairs ----
t1_mni = sorted(OUT_MNI.glob("*_T1w_MNI.nii.gz"))
mask_mni_by_base = {p.name.replace("_lesion_mask_MNI.nii.gz",""): p
                    for p in OUT_MNI.glob("*_lesion_mask_MNI.nii.gz")}
PAIRS = {}
for t1p in t1_mni:
    key  = _key_from_name(t1p)
    base = t1p.name.replace("_T1w_MNI.nii.gz", "")
    mskp = mask_mni_by_base.get(base)
    if mskp: PAIRS[key] = {"t1_mni": t1p, "mask_mni": mskp}

assert PAIRS, f"No *_T1w_MNI / *_lesion_mask_MNI pairs found in {OUT_MNI}"

# ---- MNI audit ----
bad = []
for k, v in PAIRS.items():
    ti = _load_img(str(v["t1_mni"])); mi = _load_img(str(v["mask_mni"]))
    ok_shape = (ti.shape[:3] == tpl_shape) and (mi.shape[:3] == tpl_shape)
    ok_zooms = (_zooms3(ti) == tpl_zooms) and (_zooms3(mi) == tpl_zooms)
    ok_aff_t1  = _affine_equal(ti.affine, tpl_aff)
    ok_aff_msk = _affine_equal(mi.affine, tpl_aff)
    if not (ok_shape and ok_zooms and ok_aff_t1 and ok_aff_msk):
        bad.append(dict(
            key=k, t1=v["t1_mni"].name, msk=v["mask_mni"].name,
            t1_shape=ti.shape[:3], msk_shape=mi.shape[:3],
            t1_zooms=_zooms3(ti), msk_zooms=_zooms3(mi),
            t1_aff_ok=ok_aff_t1, msk_aff_ok=ok_aff_msk
        ))

print(f"[MNI audit] dir={OUT_MNI.name} | total pairs: {len(PAIRS)} | OK: {len(PAIRS)-len(bad)} | FAIL: {len(bad)}")
if bad:
    for row in bad[:10]: print(row)

# ---- Optional native view wiring (if you created resampled masks) ----
if HAVE_NATIVE:
    for k in list(PAIRS.keys()):
        t1_native = next(iter(ROOT.glob(f"{k}*_T1w.nii.gz")), None)
        m_native  = OUT_NATMSK / f"{k}_lesion_mask_T1w_native.nii.gz"
        if t1_native and m_native.exists():
            PAIRS[k]["t1_native"] = t1_native
            PAIRS[k]["mask_t1"]   = m_native

# ---- Viewer ----
keys_sorted = sorted(PAIRS.keys())
pair_dd   = W.Dropdown(options=keys_sorted, description="Case:", layout=W.Layout(width="100%"))
slice_sl  = W.IntSlider(description="Axial slice:", min=0, max=1, value=0, continuous_update=False, layout=W.Layout(width="60%"))
alpha_sl  = W.FloatSlider(description="Mask α:", min=0.0, max=1.0, step=0.05, value=0.55, layout=W.Layout(width="35%"))
edges_cb  = W.Checkbox(description="Edges only", value=True)
invert_cb = W.Checkbox(description="Invert image", value=False)

def _bg_options_for(key: str):
    opts = [("T1w_MNI (final)", "MNI")]
    if "t1_native" in PAIRS[key] and "mask_t1" in PAIRS[key]:
        opts.append(("T1w_native (with native mask)", "NATIVE"))
    return opts

bg_radio = W.RadioButtons(options=_bg_options_for(keys_sorted[0]), value="MNI",
                          description="Background:", layout=W.Layout(width="40%"))

status = W.HTML(f"<b>Viewer</b> — cases: {len(keys_sorted)}")
controls = W.VBox([status, pair_dd, W.HBox([slice_sl, alpha_sl]), W.HBox([edges_cb, invert_cb, bg_radio])])
out = W.Output()

def _update_bg_options(*_):
    key = pair_dd.value
    bg_radio.options = _bg_options_for(key)
    if bg_radio.value not in [v for _, v in bg_radio.options]:
        bg_radio.value = bg_radio.options[0][1]

def _update_slice_range(*_):
    key = pair_dd.value
    if bg_radio.value == "MNI":
        vol = _load_vol(str(PAIRS[key]["t1_mni"]))
    else:
        vol = _load_vol(str(PAIRS[key]["t1_native"]))
    slice_sl.max = max(0, int(vol.shape[2] - 1))
    slice_sl.value = min(slice_sl.value, slice_sl.max)

def _draw(*_):
    with out:
        clear_output(wait=True)
        try:
            key = pair_dd.value
            if bg_radio.value == "MNI":
                img_p  = PAIRS[key]["t1_mni"];  mask_p = PAIRS[key]["mask_mni"];  bg_name = "T1w_MNI"
                ti = _load_img(str(img_p)); mi = _load_img(str(mask_p))
                aff_ok = _affine_equal(ti.affine, tpl_aff) and _affine_equal(mi.affine, tpl_aff)
                shp_ok = (ti.shape[:3] == tpl_shape) and (mi.shape[:3] == tpl_shape)
                z_ok   = (_zooms3(ti) == tpl_zooms) and (_zooms3(mi) == tpl_zooms)
                audit = f" | MNI check: shape {'✅' if shp_ok else '⚠️'}, zooms {'✅' if z_ok else '⚠️'}, affine {'✅' if aff_ok else '⚠️'}"
            else:
                img_p  = PAIRS[key]["t1_native"]; mask_p = PAIRS[key]["mask_t1"];  bg_name = "T1w_native"; audit = ""

            img_hdr = _load_img(str(img_p)); img = _load_vol(str(img_p))
            msk_hdr = _load_img(str(mask_p)); msk = (_load_vol(str(mask_p)) > 0.5)

            z_img = _zooms3(img_hdr); z_msk = _zooms3(msk_hdr)
            aff_eq = _affine_equal(img_hdr.affine, msk_hdr.affine)
            shp_eq = img_hdr.shape[:3] == msk_hdr.shape[:3]
            status.value = (f"<b>Viewer</b> — cases: {len(keys_sorted)} | {bg_name}: "
                            f"shape {img_hdr.shape[:3]} vs mask {msk_hdr.shape[:3]} "
                            f"| zooms {tuple(round(v,3) for v in z_img)} / {tuple(round(v,3) for v in z_msk)} "
                            f"| affines match: {'✅' if aff_eq else '⚠️'} | shapes match: {'✅' if shp_eq else '⚠️'}{audit}")

            img_view = _normalize(img.copy())
            if invert_cb.value: img_view = 1.0 - img_view
            idx = int(slice_sl.value)
            img2d = img_view[:, :, idx]; m2d = msk[:, :, idx]

            plt.figure(figsize=(5.6, 5.6))
            plt.imshow(img2d.T, cmap="gray", origin="lower")
            if edges_cb.value:
                plt.contour(_edges2d(m2d).T, levels=[0.5], linewidths=0.8, colors="r")
            else:
                plt.imshow(np.ma.masked_where(~m2d.T, m2d.T), cmap="jet", alpha=float(alpha_sl.value), origin="lower")
            plt.axis("off"); plt.tight_layout(); plt.show(); plt.close()

            print(f"Image: {img_p.name}\nMask : {mask_p.name}\nDir: {OUT_MNI}")

        except Exception as exc:
            print("Draw error:", exc)

def _refresh(*_):
    _update_bg_options(); _update_slice_range(); _draw()

pair_dd.observe(_refresh, names="value")
slice_sl.observe(_draw, names="value")
alpha_sl.observe(_draw, names="value")
edges_cb.observe(_draw, names="value")
invert_cb.observe(_draw, names="value")
bg_radio.observe(_refresh, names="value")

_refresh()
display(controls, out)


ROOT: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks
Found OUT_MNI: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed
[MNI audit] dir=mni_1mm_ants_fixed | total pairs: 203 | OK: 203 | FAIL: 0


Output()

In [9]:
from pathlib import Path
import pandas as pd, numpy as np

OUT = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed")
qc = pd.read_csv(OUT / "qc_summary.csv")

# mask volume (mm^3) from header zooms * voxel count
def parse_tuple(s):
    # strings like "(193, 229, 193)" or "(1.0, 1.0, 1.0)"
    s = s.strip().strip("()")
    return tuple(float(x) for x in s.split(","))
vol_mm3 = []
for z_str, nvox in zip(qc["mask_zooms"], qc["mask_nonzero"]):
    z = parse_tuple(z_str)
    vol_mm3.append(nvox * (z[0]*z[1]*z[2]))
qc["mask_vol_mm3"] = vol_mm3

print("Pairs:", len(qc))
print("Mask voxel counts — min/median/max:", int(qc["mask_nonzero"].min()),
      int(qc["mask_nonzero"].median()), int(qc["mask_nonzero"].max()))
print("Mask volume (ml) — min/median/max:",
      round(qc["mask_vol_mm3"].min()/1000,2),
      round(qc["mask_vol_mm3"].median()/1000,2),
      round(qc["mask_vol_mm3"].max()/1000,2))

# flag anything suspiciously tiny/huge
sus = qc[(qc["mask_vol_mm3"] < 1_000) | (qc["mask_vol_mm3"] > 200_000)]  # <1 ml or >200 ml
print("\nSuspicious volumes:", len(sus))
print(sus[["key","mask_nonzero","mask_vol_mm3"]].head(10).to_string(index=False))


Pairs: 203
Mask voxel counts — min/median/max: 0 50769 211755
Mask volume (ml) — min/median/max: 0.0 50.77 211.76

Suspicious volumes: 7
               key  mask_nonzero  mask_vol_mm3
 sub-M2131_ses-472        211755      211755.0
sub-M2141_ses-2539           235         235.0
sub-M2145_ses-1616           514         514.0
 sub-M2149_ses-677           232         232.0
sub-M2150_ses-2081           810         810.0
 sub-M2201_ses-294           694         694.0
 sub-M2269_ses-767             0           0.0


# couple of sus masks after the transformation

In [17]:
from pathlib import Path
import pandas as pd, nibabel as nib, numpy as np, matplotlib.pyplot as plt

ROOT = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks")
OUT_MNI = ROOT / "mni_1mm_ants_fixed"
QC_CSV = OUT_MNI / "qc_summary.csv"

# the suspicious keys you found
sus_keys = [
    "sub-M2131_ses-472",
    "sub-M2141_ses-2539",
    "sub-M2145_ses-1616",
    "sub-M2149_ses-677",
    "sub-M2150_ses-2081",
    "sub-M2201_ses-294",
    "sub-M2269_ses-767",
]

df = pd.read_csv(QC_CSV)
sus = df[df["key"].isin(sus_keys)].copy()

def count_native_voxels(path):
    if not Path(path).exists():
        return None
    d = nib.load(path).get_fdata()
    return int(np.count_nonzero(d > 0.5))

def count_border_voxels(mask_nii):
    d = (nib.load(mask_nii).get_fdata() > 0.5)
    z,y,x = d.shape
    border = d[0,:,:].sum() + d[z-1,:,:].sum() + d[:,0,:].sum() + d[:,y-1,:].sum() + d[:,:,0].sum() + d[:,:,x-1].sum()
    return int(border)

# compute extra metrics
nat_vox, mni_border = [], []
for _, r in sus.iterrows():
    nat_vox.append(count_native_voxels(r["mask_native"]))
    mni_border.append(count_border_voxels(r["mask_mni"]))
sus["native_mask_nonzero"] = nat_vox
sus["mni_border_touching"] = mni_border

print(sus[["key","native_mask_nonzero","mask_nonzero","mni_border_touching"]].to_string(index=False))

# Make quick PNGs per case (native vs MNI)
def norm01(a):
    a = a.astype(np.float32)
    nz = a[a>0]
    if nz.size==0: return np.zeros_like(a, np.float32)
    p1,p99 = np.percentile(nz,[1,99])
    a = np.clip(a,p1,p99)
    a = (a - a.min()) / (a.max() - a.min() + 1e-6)
    return a

for _, r in sus.iterrows():
    key = r["key"]

    # native
    t1_nat = r["t1_src"]                 # native T1 path
    msk_nat= r["mask_native"]            # native-resampled mask path
    # MNI
    t1_mni = r["t1_mni"]
    msk_mni= r["mask_mni"]

    if not (Path(t1_nat).exists() and Path(msk_nat).exists() and Path(t1_mni).exists() and Path(msk_mni).exists()):
        print(f"[skip] missing file(s) for {key}")
        continue

    nat_img = nib.load(t1_nat).get_fdata()
    nat_msk = (nib.load(msk_nat).get_fdata() > 0.5)

    mni_img = nib.load(t1_mni).get_fdata()
    mni_msk = (nib.load(msk_mni).get_fdata() > 0.5)

    # middle axial slice (by mask bbox if present, else center)
    def mid_slice(msk, shape):
        if msk.any():
            idx = np.argwhere(msk)
            zmin,zmax = idx[:,2].min(), idx[:,2].max()
            return int((zmin+zmax)//2)
        return shape[2]//2

    zn = mid_slice(nat_msk, nat_img.shape)
    zm = mid_slice(mni_msk, mni_img.shape)

    fig = plt.figure(figsize=(8,4.5))

    ax1 = fig.add_subplot(1,2,1)
    ax1.imshow(norm01(nat_img[:,:,zn]).T, cmap="gray", origin="lower")
    ax1.contour(nat_msk[:,:,zn].T, levels=[0.5], linewidths=0.8, colors="r")
    ax1.set_title(f"{key}\nNATIVE (slice {zn})")
    ax1.axis("off")

    ax2 = fig.add_subplot(1,2,2)
    ax2.imshow(norm01(mni_img[:,:,zm]).T, cmap="gray", origin="lower")
    ax2.contour(mni_msk[:,:,zm].T, levels=[0.5], linewidths=0.8, colors="r")
    ax2.set_title(f"MNI (slice {zm})\nBorder-touch: {count_border_voxels(r['mask_mni'])}")
    ax2.axis("off")

    fig.tight_layout()
    out_png = OUT_MNI / f"{key}_audit.png"
    plt.savefig(out_png, dpi=140); plt.close()
    print("Saved", out_png)


               key  native_mask_nonzero  mask_nonzero  mni_border_touching
 sub-M2131_ses-472               363885        211755                    0
sub-M2141_ses-2539                  203           235                    0
sub-M2145_ses-1616                  440           514                    0
 sub-M2149_ses-677                  238           232                    0
sub-M2150_ses-2081                  701           810                    0
 sub-M2201_ses-294                  532           694                    0
 sub-M2269_ses-767                    0             0                    0
Saved /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/sub-M2131_ses-472_audit.png
Saved /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/sub-M2141_ses-2539_audit.png
Saved /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/sub-M2145_ses-1616_audit.png
Saved /home/rbielski/ARC/ds004884/derivat

# Compare the native vs MNI transformed to see if the transformation messed things up or if it was like that before

In [18]:
from pathlib import Path
import pandas as pd, nibabel as nib, numpy as np

ROOT = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks")
OUT  = ROOT / "mni_1mm_ants_fixed"
qc   = pd.read_csv(OUT / "qc_summary.csv")

sus = ["sub-M2131_ses-472","sub-M2141_ses-2539","sub-M2145_ses-1616","sub-M2149_ses-677",
       "sub-M2150_ses-2081","sub-M2201_ses-294","sub-M2269_ses-767"]
df = qc[qc["key"].isin(sus)].copy()

def vol_ml(nii_path):
    img = nib.load(nii_path)
    vox = (img.get_fdata() > 0.5).sum()
    z = img.header.get_zooms()[:3]
    return (vox * z[0]*z[1]*z[2]) / 1000.0

rows = []
for _, r in df.iterrows():
    rows.append({
        "key": r["key"],
        "native_mask_ml": vol_ml(r["mask_native"]) if Path(r["mask_native"]).exists() else None,
        "mni_mask_ml":    vol_ml(r["mask_mni"])    if Path(r["mask_mni"]).exists()    else None,
    })
out = pd.DataFrame(rows)
out["ratio_MNI/native"] = (out["mni_mask_ml"] / out["native_mask_ml"]).round(3)
print(out.to_string(index=False))


               key  native_mask_ml  mni_mask_ml  ratio_MNI/native
 sub-M2131_ses-472         363.885      211.755             0.582
sub-M2141_ses-2539           0.203        0.235             1.158
sub-M2145_ses-1616           0.440        0.514             1.168
 sub-M2149_ses-677           0.238        0.232             0.975
sub-M2150_ses-2081           0.701        0.810             1.155
 sub-M2201_ses-294           0.532        0.694             1.305
 sub-M2269_ses-767           0.000        0.000               NaN


# now look at ALL sizes

In [23]:
# === ARC T1w + lesion-masks: metrics + rich summaries + sample rows (native & MNI) ===
from pathlib import Path
import re
import numpy as np
import pandas as pd
import nibabel as nib
from math import prod
from IPython.display import display

# ---- Directories ----
ROOT    = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks")
OUT_NAT = ROOT / "native_resampled_masks"   # *_lesion_mask_T1w_native.nii.gz
OUT_MNI = ROOT / "mni_1mm_ants_fixed"       # *_T1w_MNI.nii.gz + *_lesion_mask_MNI.nii.gz
assert OUT_NAT.exists(), f"Missing {OUT_NAT}"
assert OUT_MNI.exists(), f"Missing {OUT_MNI}"

# ---- Helpers ----
def _tag(s, t):
    m = re.search(fr"({t}-[^_]+)", s)
    return m.group(1) if m else None

def key_from_name(p: Path) -> str:
    sub = _tag(p.name, "sub")
    ses = _tag(p.name, "ses")
    return "_".join([x for x in (sub, ses) if x])

def t1_pref(name: str) -> int:
    n = name.lower()
    if "tfl" in n: return 0
    if "mprage" in n: return 1
    if "mp2rage" in n: return 2
    return 3

def choose_native_t1(key: str) -> Path | None:
    cands = sorted(ROOT.glob(f"{key}*_T1w.nii.gz"))
    if not cands: return None
    cands.sort(key=lambda p: (t1_pref(p.name), p.name))
    return cands[0]

def img_metrics(img_path: Path, nonzero_thresh=0.0):
    img = nib.load(str(img_path))
    shp = img.shape[:3]
    z   = tuple(float(v) for v in img.header.get_zooms()[:3])
    data = img.get_fdata()
    nz  = int(np.count_nonzero(data > nonzero_thresh))
    vox_total = int(prod(shp))
    fov_mm = (shp[0]*z[0], shp[1]*z[1], shp[2]*z[2])
    voxel_vol_mm3 = z[0]*z[1]*z[2]
    fov_vol_ml = (vox_total * voxel_vol_mm3) / 1000.0
    return dict(
        shape_x=shp[0], shape_y=shp[1], shape_z=shp[2],
        zooms_x=z[0], zooms_y=z[1], zooms_z=z[2],
        vox_total=vox_total,
        img_nonzero=nz,
        brain_frac=(nz/vox_total if vox_total else np.nan),
        fov_mm_x=fov_mm[0], fov_mm_y=fov_mm[1], fov_mm_z=fov_mm[2],
        voxel_vol_mm3=voxel_vol_mm3,
        fov_vol_ml=fov_vol_ml,
    )

def mask_metrics(mask_path: Path, zooms_xyz: tuple[float,float,float]):
    m = nib.load(str(mask_path))
    vox = int(np.count_nonzero(m.get_fdata() > 0.5))
    ml  = (vox * zooms_xyz[0] * zooms_xyz[1] * zooms_xyz[2]) / 1000.0
    return dict(mask_voxels=vox, mask_ml=ml)

# ---- Build rows driven by MNI outputs ----
t1_mni_files = sorted(OUT_MNI.glob("*_T1w_MNI.nii.gz"))
assert t1_mni_files, f"No *_T1w_MNI.nii.gz found in {OUT_MNI}"

rows = []
for t1_mni in t1_mni_files:
    key = key_from_name(t1_mni)
    msk_mni = OUT_MNI / f"{key}_lesion_mask_MNI.nii.gz"
    if not msk_mni.exists():
        continue

    # Native counterparts
    t1_nat = choose_native_t1(key)
    msk_nat = OUT_NAT / f"{key}_lesion_mask_T1w_native.nii.gz"

    # MNI metrics
    mni_img = img_metrics(t1_mni, nonzero_thresh=0.0)
    mni_mask = mask_metrics(msk_mni, (mni_img["zooms_x"], mni_img["zooms_y"], mni_img["zooms_z"]))

    # Native metrics
    if t1_nat and msk_nat.exists():
        nat_img = img_metrics(t1_nat, nonzero_thresh=0.0)
        nat_mask = mask_metrics(msk_nat, (nat_img["zooms_x"], nat_img["zooms_y"], nat_img["zooms_z"]))
    else:
        nat_img = {k: np.nan for k in [
            "shape_x","shape_y","shape_z","zooms_x","zooms_y","zooms_z","vox_total",
            "img_nonzero","brain_frac","fov_mm_x","fov_mm_y","fov_mm_z","voxel_vol_mm3","fov_vol_ml"
        ]}
        nat_mask = {"mask_voxels": np.nan, "mask_ml": np.nan}
        t1_nat = None
        msk_nat = None

    rows.append({
        "key": key,
        # Native
        "nat_shape_x": nat_img["shape_x"], "nat_shape_y": nat_img["shape_y"], "nat_shape_z": nat_img["shape_z"],
        "nat_zooms_x": nat_img["zooms_x"], "nat_zooms_y": nat_img["zooms_y"], "nat_zooms_z": nat_img["zooms_z"],
        "nat_vox_total": nat_img["vox_total"], "nat_img_nonzero": nat_img["img_nonzero"], "nat_brain_frac": nat_img["brain_frac"],
        "nat_fov_mm_x": nat_img["fov_mm_x"], "nat_fov_mm_y": nat_img["fov_mm_y"], "nat_fov_mm_z": nat_img["fov_mm_z"],
        "nat_voxel_vol_mm3": nat_img["voxel_vol_mm3"], "nat_fov_vol_ml": nat_img["fov_vol_ml"],
        "nat_mask_voxels": nat_mask["mask_voxels"], "nat_mask_ml": nat_mask["mask_ml"],
        # MNI
        "mni_shape_x": mni_img["shape_x"], "mni_shape_y": mni_img["shape_y"], "mni_shape_z": mni_img["shape_z"],
        "mni_zooms_x": mni_img["zooms_x"], "mni_zooms_y": mni_img["zooms_y"], "mni_zooms_z": mni_img["zooms_z"],
        "mni_vox_total": mni_img["vox_total"], "mni_img_nonzero": mni_img["img_nonzero"], "mni_brain_frac": mni_img["brain_frac"],
        "mni_fov_mm_x": mni_img["fov_mm_x"], "mni_fov_mm_y": mni_img["fov_mm_y"], "mni_fov_mm_z": mni_img["fov_mm_z"],
        "mni_voxel_vol_mm3": mni_img["voxel_vol_mm3"], "mni_fov_vol_ml": mni_img["fov_vol_ml"],
        "mni_mask_voxels": mni_mask["mask_voxels"], "mni_mask_ml": mni_mask["mask_ml"],
    })

df = pd.DataFrame(rows).sort_values("key").reset_index(drop=True)

# Save CSV
csv_path = OUT_MNI / "image_mask_metrics.csv"
df.to_csv(csv_path, index=False)
print(f"Wrote: {csv_path}")
print("Pairs:", len(df))

# ===== Summaries =====
def summary_block(prefix: str, label: str):
    ok = df[f"{prefix}_img_nonzero"].notna()
    if not ok.any():
        print(f"\n=== {label} ===\n(no data)")
        return
    vox   = df.loc[ok, f"{prefix}_img_nonzero"].to_numpy(int)
    total = df.loc[ok, f"{prefix}_vox_total"].to_numpy(int)
    frac  = df.loc[ok, f"{prefix}_brain_frac"].to_numpy(float)
    fovml = df.loc[ok, f"{prefix}_fov_vol_ml"].to_numpy(float)
    mvox  = df.loc[ok, f"{prefix}_mask_voxels"].to_numpy(int)
    mml   = df.loc[ok, f"{prefix}_mask_ml"].to_numpy(float)

    pct = lambda a, q: np.percentile(a, q)
    print(f"\n=== {label} ===")
    print("ICV proxy (nonzero voxels):")
    print(f"  min/median/max: {vox.min():,} / {int(pct(vox,50)):,} / {vox.max():,}")
    print("Brain fraction of FOV (nonzero/total):")
    print(f"  mean±sd: {frac.mean():.3f} ± {frac.std():.3f} | p10/50/90: {pct(frac,10):.3f} / {pct(frac,50):.3f} / {pct(frac,90):.3f}")
    print("FOV volume (mL):")
    print(f"  min/median/max: {fovml.min():.1f} / {pct(fovml,50):.1f} / {fovml.max():.1f}")
    print("Lesion mask volume (mL):")
    print(f"  min/median/max: {mml.min():.2f} / {pct(mml,50):.2f} / {mml.max():.2f}")
    print("Lesion mask voxels:")
    print(f"  min/median/max: {mvox.min():,} / {int(pct(mvox,50)):,} / {mvox.max():,}")

summary_block("nat", "NATIVE summary")
summary_block("mni", "MNI summary")

# ===== Sample tables (first 10 rows) =====
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

native_cols = [
    "key",
    "nat_shape_x","nat_shape_y","nat_shape_z",
    "nat_zooms_x","nat_zooms_y","nat_zooms_z",
    "nat_vox_total","nat_img_nonzero","nat_brain_frac",
    "nat_fov_mm_x","nat_fov_mm_y","nat_fov_mm_z","nat_fov_vol_ml",
    "nat_mask_voxels","nat_mask_ml",
]
mni_cols = [
    "key",
    "mni_shape_x","mni_shape_y","mni_shape_z",
    "mni_zooms_x","mni_zooms_y","mni_zooms_z",
    "mni_vox_total","mni_img_nonzero","mni_brain_frac",
    "mni_fov_mm_x","mni_fov_mm_y","mni_fov_mm_z","mni_fov_vol_ml",
    "mni_mask_voxels","mni_mask_ml",
]
combined_cols = [
    "key",
    "nat_shape_x","nat_shape_y","nat_shape_z","nat_zooms_x","nat_zooms_y","nat_zooms_z",
    "nat_vox_total","nat_img_nonzero","nat_mask_voxels","nat_mask_ml",
    "mni_shape_x","mni_shape_y","mni_shape_z","mni_zooms_x","mni_zooms_y","mni_zooms_z",
    "mni_vox_total","mni_img_nonzero","mni_mask_voxels","mni_mask_ml"
]

print("\n=== NATIVE (first 10 rows) ===")
display(df[native_cols].head(10))
print("\n=== MNI (first 10 rows) ===")
display(df[mni_cols].head(10))
print("\n=== COMBINED (first 10 rows) ===")
display(df[combined_cols].head(10))


Wrote: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/image_mask_metrics.csv
Pairs: 203

=== NATIVE summary ===
ICV proxy (nonzero voxels):
  min/median/max: 2,264,859 / 2,843,324 / 3,687,626
Brain fraction of FOV (nonzero/total):
  mean±sd: 0.230 ± 0.027 | p10/50/90: 0.195 / 0.230 / 0.267
FOV volume (mL):
  min/median/max: 10485.8 / 12582.9 / 12582.9
Lesion mask volume (mL):
  min/median/max: 0.00 / 67.35 / 363.88
Lesion mask voxels:
  min/median/max: 0 / 67,354 / 363,885

=== MNI summary ===
ICV proxy (nonzero voxels):
  min/median/max: 1,988,864 / 2,056,336 / 2,293,411
Brain fraction of FOV (nonzero/total):
  mean±sd: 0.242 ± 0.005 | p10/50/90: 0.238 / 0.241 / 0.248
FOV volume (mL):
  min/median/max: 8530.0 / 8530.0 / 8530.0
Lesion mask volume (mL):
  min/median/max: 0.00 / 50.77 / 211.75
Lesion mask voxels:
  min/median/max: 0 / 50,769 / 211,755

=== NATIVE (first 10 rows) ===


,key,nat_shape_x,nat_shape_y,nat_shape_z,nat_zooms_x,nat_zooms_y,nat_zooms_z,nat_vox_total,nat_img_nonzero,nat_brain_frac,nat_fov_mm_x,nat_fov_mm_y,nat_fov_mm_z,nat_fov_vol_ml,nat_mask_voxels,nat_mask_ml
0,sub-M2012_ses-1158,160,256,256,1.0,1.0,1.0,10485760,2785721,0.265667,160.0,256.0,256.0,10485.76,64101,64.101
1,sub-M2018_ses-341,160,256,256,1.0,1.0,1.0,10485760,2794148,0.266471,160.0,256.0,256.0,10485.76,46626,46.626
2,sub-M2026_ses-3098,160,256,256,1.0,1.0,1.0,10485760,2651291,0.252847,160.0,256.0,256.0,10485.76,6798,6.798
3,sub-M2029_ses-180,160,256,256,1.0,1.0,1.0,10485760,2767384,0.263918,160.0,256.0,256.0,10485.76,153804,153.804
4,sub-M2034_ses-1568,160,256,256,1.0,1.0,1.0,10485760,3086562,0.294357,160.0,256.0,256.0,10485.76,229376,229.376
5,sub-M2035_ses-4295,160,256,256,1.0,1.0,1.0,10485760,2577108,0.245772,160.0,256.0,256.0,10485.76,131306,131.306
6,sub-M2036_ses-695,160,256,256,1.0,1.0,1.0,10485760,3127415,0.298254,160.0,256.0,256.0,10485.76,128308,128.308
7,sub-M2037_ses-275,160,256,256,1.0,1.0,1.0,10485760,2365214,0.225564,160.0,256.0,256.0,10485.76,35596,35.596
8,sub-M2042_ses-1953,160,256,256,1.0,1.0,1.0,10485760,3354619,0.319921,160.0,256.0,256.0,10485.76,177622,177.622
9,sub-M2043_ses-183,160,256,256,1.0,1.0,1.0,10485760,2919390,0.278415,160.0,256.0,256.0,10485.76,4259,4.259



=== MNI (first 10 rows) ===


,key,mni_shape_x,mni_shape_y,mni_shape_z,mni_zooms_x,mni_zooms_y,mni_zooms_z,mni_vox_total,mni_img_nonzero,mni_brain_frac,mni_fov_mm_x,mni_fov_mm_y,mni_fov_mm_z,mni_fov_vol_ml,mni_mask_voxels,mni_mask_ml
0,sub-M2012_ses-1158,193,229,193,1.0,1.0,1.0,8530021,2106241,0.246921,193.0,229.0,193.0,8530.021,52007,52.007
1,sub-M2018_ses-341,193,229,193,1.0,1.0,1.0,8530021,2205956,0.258611,193.0,229.0,193.0,8530.021,38023,38.023
2,sub-M2026_ses-3098,193,229,193,1.0,1.0,1.0,8530021,2136067,0.250418,193.0,229.0,193.0,8530.021,8560,8.560
3,sub-M2029_ses-180,193,229,193,1.0,1.0,1.0,8530021,2140350,0.250920,193.0,229.0,193.0,8530.021,104897,104.897
4,sub-M2034_ses-1568,193,229,193,1.0,1.0,1.0,8530021,2098494,0.246013,193.0,229.0,193.0,8530.021,121398,121.398
5,sub-M2035_ses-4295,193,229,193,1.0,1.0,1.0,8530021,2116388,0.248111,193.0,229.0,193.0,8530.021,75186,75.186
6,sub-M2036_ses-695,193,229,193,1.0,1.0,1.0,8530021,2107542,0.247073,193.0,229.0,193.0,8530.021,70103,70.103
7,sub-M2037_ses-275,193,229,193,1.0,1.0,1.0,8530021,2246780,0.263397,193.0,229.0,193.0,8530.021,42415,42.415
8,sub-M2042_ses-1953,193,229,193,1.0,1.0,1.0,8530021,2067419,0.242370,193.0,229.0,193.0,8530.021,77424,77.424
9,sub-M2043_ses-183,193,229,193,1.0,1.0,1.0,8530021,2106860,0.246994,193.0,229.0,193.0,8530.021,4427,4.427



=== COMBINED (first 10 rows) ===


,key,nat_shape_x,nat_shape_y,nat_shape_z,nat_zooms_x,nat_zooms_y,nat_zooms_z,nat_vox_total,nat_img_nonzero,nat_mask_voxels,nat_mask_ml,mni_shape_x,mni_shape_y,mni_shape_z,mni_zooms_x,mni_zooms_y,mni_zooms_z,mni_vox_total,mni_img_nonzero,mni_mask_voxels,mni_mask_ml
0,sub-M2012_ses-1158,160,256,256,1.0,1.0,1.0,10485760,2785721,64101,64.101,193,229,193,1.0,1.0,1.0,8530021,2106241,52007,52.007
1,sub-M2018_ses-341,160,256,256,1.0,1.0,1.0,10485760,2794148,46626,46.626,193,229,193,1.0,1.0,1.0,8530021,2205956,38023,38.023
2,sub-M2026_ses-3098,160,256,256,1.0,1.0,1.0,10485760,2651291,6798,6.798,193,229,193,1.0,1.0,1.0,8530021,2136067,8560,8.560
3,sub-M2029_ses-180,160,256,256,1.0,1.0,1.0,10485760,2767384,153804,153.804,193,229,193,1.0,1.0,1.0,8530021,2140350,104897,104.897
4,sub-M2034_ses-1568,160,256,256,1.0,1.0,1.0,10485760,3086562,229376,229.376,193,229,193,1.0,1.0,1.0,8530021,2098494,121398,121.398
5,sub-M2035_ses-4295,160,256,256,1.0,1.0,1.0,10485760,2577108,131306,131.306,193,229,193,1.0,1.0,1.0,8530021,2116388,75186,75.186
6,sub-M2036_ses-695,160,256,256,1.0,1.0,1.0,10485760,3127415,128308,128.308,193,229,193,1.0,1.0,1.0,8530021,2107542,70103,70.103
7,sub-M2037_ses-275,160,256,256,1.0,1.0,1.0,10485760,2365214,35596,35.596,193,229,193,1.0,1.0,1.0,8530021,2246780,42415,42.415
8,sub-M2042_ses-1953,160,256,256,1.0,1.0,1.0,10485760,3354619,177622,177.622,193,229,193,1.0,1.0,1.0,8530021,2067419,77424,77.424
9,sub-M2043_ses-183,160,256,256,1.0,1.0,1.0,10485760,2919390,4259,4.259,193,229,193,1.0,1.0,1.0,8530021,2106860,4427,4.427


# Rundown

FOV = Field of View — literally the 3D grid stored: (shape_x, shape_y, shape_z) voxels with zooms mm/voxel.

vox_total = shape_x * shape_y * shape_z

brain_frac = img_nonzero / vox_total is a crude “how much of the grid has signal”.

Native vs MNI differences are expected:

Your native T1s come in different matrices and voxel sizes (some ~0.8 mm^3, others different), and they include head/neck/scalp signal → higher img_nonzero and more variability. That’s why native ICV (nonzero voxel counts) ranged from ~2.26M to ~3.69M and brain_frac ≈ 0.23 ± 0.027.

Your MNI outputs were written on a standard, brain-only template (TemplateFlow tpl-MNI152NLin2009cAsym, desc=brain). The background outside the brain is zero by design, the grid is consistent, and the voxel size is 1 mm isotropic. Hence much tighter nonzero counts (≈2.0–2.29M) and brain_frac ≈ 0.242 ± 0.005.

“Did we go down in resolution?”
No. Final outputs are 1 mm iso in MNI space. Some native scans likely had sub-millimeter voxels (e.g., 0.8 mm), so if you compare voxel counts you might see fewer voxels after resampling to 1 mm for the same physical volume. That’s not a loss of spatial detail in the final standard space; it’s just a different grid. (Also, our logs intentionally used a 2 mm template for registration speed — res-02 — and then applied transforms to 1 mm — res-01. That speeds SyN but keeps your final outputs at 1 mm.)

ICV here is “nonzero voxels,” not a brain mask.
It’s a quick proxy. If you want a truer ICV, we’d run a brain extraction (or tissue segmentation) and count those voxels instead. For your current QC needs, img_nonzero is fine to compare coverage/consistency.

We’ll now standardize:

apply a common MNI brain mask per case

robust intensity normalization inside the brain

component-aware mask cleanup (optional, conservative)

write a QC CSV with before/after stats

In [25]:
# === Common preprocessing utilities for MNI-registered T1w + lesion masks ===
import sys, os, json, csv, shutil, subprocess, site
from pathlib import Path
import numpy as np
import nibabel as nib
from nibabel.processing import resample_from_to
import pandas as pd
from scipy.ndimage import label

# --- TemplateFlow brain mask (MNI152NLin2009cAsym, res-01) ---
try:
    from templateflow.api import get as tf_get
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "--user", "templateflow"], check=True)
    sys.path.append(site.getusersitepackages())
    from templateflow.api import get as tf_get

TPL_MASK = tf_get("MNI152NLin2009cAsym", resolution=1, suffix="mask", desc="brain", extension="nii.gz")
TPL_MASK = Path(TPL_MASK[0] if isinstance(TPL_MASK, (list, tuple)) else TPL_MASK)
assert TPL_MASK.exists(), f"Missing TemplateFlow brain mask: {TPL_MASK}"

# --- Helpers ---
def _load(path: Path) -> nib.Nifti1Image:
    return nib.load(str(path))

def _save_like(ref_img: nib.Nifti1Image, data: np.ndarray, out_path: Path, dtype=np.float32):
    img = nib.Nifti1Image(data.astype(dtype), ref_img.affine, ref_img.header)
    img.set_data_dtype(dtype)
    nib.save(img, str(out_path))

def resample_brain_mask_to(ref_img: nib.Nifti1Image) -> np.ndarray:
    """Resample the TemplateFlow brain mask to the target grid (order=0)."""
    tpl = _load(TPL_MASK)
    rs  = resample_from_to(tpl, (ref_img.shape, ref_img.affine), order=0)
    return (rs.get_fdata() > 0.5)

def robust_scale_in_brain(t1_img: nib.Nifti1Image, brain_mask: np.ndarray, clip=(1,99)) -> np.ndarray:
    """Clip to [p1,p99] inside brain, then scale to [0,1]. Outside-brain set to 0."""
    x = t1_img.get_fdata().astype(np.float32)
    x[~brain_mask] = 0.0
    brain_vals = x[brain_mask]
    if brain_vals.size == 0:
        return np.zeros_like(x, dtype=np.float32)
    p1, p99 = np.percentile(brain_vals, clip)
    x = np.clip(x, p1, p99, out=x)
    # min-max inside brain
    bmin, bmax = x[brain_mask].min(), x[brain_mask].max()
    if bmax > bmin:
        x = (x - bmin) / (bmax - bmin)
    else:
        x[:] = 0.0
    x[~brain_mask] = 0.0
    return x

def mask_binarize(mask_img: nib.Nifti1Image, thr=0.5) -> np.ndarray:
    return (mask_img.get_fdata() > thr).astype(np.uint8)

def component_stats(mask_bin: np.ndarray):
    if mask_bin.sum() == 0:
        return dict(n_components=0, total_vox=0, largest_vox=0, frac_in_largest=np.nan)
    lab, n = label(mask_bin)
    sizes = np.bincount(lab.ravel())[1:]
    total = int(sizes.sum())
    largest = int(sizes.max())
    return dict(n_components=int(n), total_vox=total, largest_vox=largest,
                frac_in_largest=float(largest/total))

def clean_components(mask_img: nib.Nifti1Image, min_vox=100) -> np.ndarray:
    m = (mask_img.get_fdata() > 0.5)
    if m.sum() == 0:
        return m.astype(np.uint8)
    lab, n = label(m)
    if n == 0:
        return m.astype(np.uint8)
    sizes = np.bincount(lab.ravel())
    keep_ids = [i for i, s in enumerate(sizes) if i != 0 and s >= min_vox]
    if not keep_ids:
        return np.zeros_like(m, dtype=np.uint8)
    keep = np.isin(lab, keep_ids)
    return keep.astype(np.uint8)

def mm3_per_voxel(img: nib.Nifti1Image) -> float:
    z = img.header.get_zooms()[:3]
    return float(z[0]*z[1]*z[2])

def qc_row(key, t1_img, t1_norm, brain_mask, msk_img, msk_clean):
    # native stats
    t1 = t1_img.get_fdata()
    nz = int(np.count_nonzero(t1))
    vox_total = int(np.prod(t1_img.shape[:3]))
    frac = nz/vox_total if vox_total else np.nan
    vmm3 = mm3_per_voxel(t1_img)
    # norm stats
    nnz = int(np.count_nonzero(t1_norm))
    # mask stats
    m_bin = (msk_img.get_fdata() > 0.5)
    m_bin_clean = msk_clean.astype(bool)
    m_vox = int(m_bin.sum())
    m_vox_clean = int(m_bin_clean.sum())
    return dict(
        key=key,
        t1_shape_x=t1_img.shape[0], t1_shape_y=t1_img.shape[1], t1_shape_z=t1_img.shape[2],
        t1_zooms=str(tuple(round(v,3) for v in t1_img.header.get_zooms()[:3])),
        vox_total=vox_total, img_nonzero=nz, brain_frac=frac,
        voxel_mm3=vmm3,
        norm_img_nonzero=nnz,
        mask_voxels=m_vox, mask_ml=(m_vox*vmm3)/1000.0,
        mask_voxels_clean=m_vox_clean, mask_ml_clean=(m_vox_clean*vmm3)/1000.0,
        brain_voxels=int(brain_mask.sum())
    )

def process_dataset(
    IN_DIR: Path,
    OUT_DIR: Path,
    key_glob="*_T1w_MNI.nii.gz",
    mask_suffix="_lesion_mask_MNI.nii.gz",
    clean_policy="component_aware",  # "none" | "all_small" | "component_aware"
    min_component_vox=100,
    tiny_ml_threshold=1.0,
    frac_major_threshold=0.8
):
    OUT_T1  = OUT_DIR / "t1_norm"
    OUT_MSK = OUT_DIR / "masks_clean"
    OUT_T1.mkdir(parents=True, exist_ok=True)
    OUT_MSK.mkdir(parents=True, exist_ok=True)

    records = []
    t1_files = sorted(IN_DIR.glob(key_glob))
    if not t1_files:
        raise RuntimeError(f"No T1s found in {IN_DIR} with pattern {key_glob}")

    def key_from(p: Path):
        s = p.name.replace("_T1w_MNI.nii.gz", "")
        # Keep ARC style "sub-XXX_ses-YYY" and ATLAS "sub-xxx_ses-1"
        return s

    for t1_path in t1_files:
        key = key_from(t1_path)
        msk_path = IN_DIR / f"{key}{mask_suffix.replace('_lesion_mask_MNI.nii.gz','')}_lesion_mask_MNI.nii.gz" \
                   if mask_suffix.endswith("_lesion_mask_MNI.nii.gz") else IN_DIR / f"{key}{mask_suffix}"
        # In both ARC and ATLAS, mask is {key}_lesion_mask_MNI.nii.gz
        msk_path = IN_DIR / f"{key}_lesion_mask_MNI.nii.gz"
        if not msk_path.exists():
            # skip if no mask
            continue

        t1_img = _load(t1_path)
        msk_img = _load(msk_path)

        # 1) Brain mask to this subject's grid
        brain_mask = resample_brain_mask_to(t1_img)

        # 2) Normalization (inside brain); set outside=0
        t1_norm = robust_scale_in_brain(t1_img, brain_mask, clip=(1,99))

        # 3) Mask cleanup (conservative, component-aware by default)
        if clean_policy == "none":
            m_clean = (msk_img.get_fdata() > 0.5).astype(np.uint8)
        elif clean_policy == "all_small":
            m_clean = clean_components(msk_img, min_vox=min_component_vox)
        else:
            # component-aware: only clean if very small & fragmented
            m_bin = (msk_img.get_fdata() > 0.5).astype(np.uint8)
            vmm3  = mm3_per_voxel(msk_img)
            total_ml = (m_bin.sum()*vmm3)/1000.0
            st = component_stats(m_bin)
            if (total_ml < tiny_ml_threshold) and (np.isfinite(st["frac_in_largest"]) and st["frac_in_largest"] < frac_major_threshold):
                m_clean = clean_components(msk_img, min_vox=min_component_vox)
            else:
                m_clean = m_bin

        # 4) Save outputs
        t1_out  = OUT_T1 / f"{key}_T1w_MNI_norm.nii.gz"
        msk_out = OUT_MSK / f"{key}_lesion_mask_MNI_clean.nii.gz"
        _save_like(t1_img, t1_norm, t1_out, dtype=np.float32)
        _save_like(msk_img, m_clean, msk_out, dtype=np.uint8)

        # 5) QC row
        rec = qc_row(key, t1_img, t1_norm, brain_mask, msk_img, m_clean)
        rec.update(component_stats((msk_img.get_fdata() > 0.5).astype(np.uint8)))
        rec.update({f"clean_{k}": v for k, v in component_stats(m_clean).items()})
        records.append(rec)

    df = pd.DataFrame(records).sort_values("key").reset_index(drop=True)
    (OUT_DIR / "preprocess_qc.csv").write_text(df.to_csv(index=False))
    print(f"Preprocess complete. Wrote: {OUT_DIR/'preprocess_qc.csv'}")
    print(f"Cases processed: {len(df)}")
    return df


100%|██████████| 160k/160k [00:00<00:00, 913kB/s] 


In [26]:
# === ARC run ===
from pathlib import Path

# Inputs (your existing MNI outputs)
ARC_MNI_IN  = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed")
# Outputs (new standardized dataset root)
ARC_OUT     = ARC_MNI_IN / "_standardized"  # creates t1_norm/ and masks_clean/

ARC_OUT.mkdir(parents=True, exist_ok=True)

df_arc = process_dataset(
    IN_DIR=ARC_MNI_IN,
    OUT_DIR=ARC_OUT,
    key_glob="*_T1w_MNI.nii.gz",
    mask_suffix="_lesion_mask_MNI.nii.gz",
    clean_policy="component_aware",  # conservative
    min_component_vox=100,           # drop components <100 vox *if* tiny & fragmented
    tiny_ml_threshold=1.0,           # only consider cleaning if total <1 ml
    frac_major_threshold=0.8         # and <80% in largest component
)

print("\nARC summary:")
print("Pairs:", len(df_arc))
print("Median lesion ml (raw/clean):",
      round(df_arc["mask_ml"].median(),2), "/",
      round(df_arc["mask_ml_clean"].median(),2))
print("Median brain_frac (raw):", round(df_arc["brain_frac"].median(),3))
print("Outputs:")
print("  T1 normalized →", (ARC_OUT / "t1_norm"))
print("  Masks cleaned →", (ARC_OUT / "masks_clean"))
print("  QC CSV        →", (ARC_OUT / "preprocess_qc.csv"))


Preprocess complete. Wrote: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/preprocess_qc.csv
Cases processed: 203

ARC summary:
Pairs: 203
Median lesion ml (raw/clean): 50.77 / 50.77
Median brain_frac (raw): 0.241
Outputs:
  T1 normalized → /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/t1_norm
  Masks cleaned → /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/masks_clean
  QC CSV        → /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/preprocess_qc.csv


In [27]:
import pandas as pd
from pathlib import Path

QC = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/preprocess_qc.csv")  # or ARC path
df = pd.read_csv(QC)

tiny_cases = df[df["mask_ml_clean"] < 1.0].sort_values("mask_ml_clean")[["key","mask_ml_clean","mask_voxels_clean"]]
print(f"Tiny cases (<1 mL): {len(tiny_cases)}")
display(tiny_cases.head(20))

# Optional: write a list to use for exclusion in training
(tiny_cases["key"]).to_csv(QC.parent / "exclude_tiny.txt", index=False, header=False)
print("Wrote:", QC.parent / "exclude_tiny.txt")


Tiny cases (<1 mL): 6


,key,mask_ml_clean,mask_voxels_clean
172,sub-M2269_ses-767,0.000,0
93,sub-M2149_ses-677,0.107,107
86,sub-M2141_ses-2539,0.235,235
90,sub-M2145_ses-1616,0.514,514
130,sub-M2201_ses-294,0.694,694
94,sub-M2150_ses-2081,0.810,810


Wrote: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/exclude_tiny.txt


# remove all of the super tiny less that 1 ml lesions into seperate folder

In [29]:
from pathlib import Path
import pandas as pd
import os, shutil

# ====== CONFIG: set this to the standardized dataset root ======
# ATLAS:
# STD_ROOT = Path("/home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized")
# ARC (uncomment to run for ARC after ATLAS):
STD_ROOT = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized")

QC_CSV   = STD_ROOT / "preprocess_qc.csv"
TINY_TXT = STD_ROOT / "exclude_tiny.txt"  # your relabeled list (one key per line)

# ====== IO helpers ======
def _link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        if dst.exists():
            dst.unlink()
        os.link(src, dst)   # hardlink (saves space/time)
    except Exception:
        shutil.copy2(src, dst)  # fallback to copy

# ====== Load keys & tiny set ======
df = pd.read_csv(QC_CSV)
all_keys = sorted(df["key"].unique())

if TINY_TXT.exists():
    tiny_keys = set(k.strip() for k in TINY_TXT.read_text().splitlines() if k.strip())
else:
    # fallback: derive tiny by volume (<1 mL) from QC
    tiny_keys = set(df.loc[df["mask_ml_clean"] < 1.0, "key"].tolist())

normal_keys = [k for k in all_keys if k not in tiny_keys]

# ====== Source files ======
SRC_T1  = STD_ROOT / "t1_norm"
SRC_MSK = STD_ROOT / "masks_clean"

# Sanity: ensure we can find files
missing = []
for k in all_keys:
    if not (SRC_T1 / f"{k}_T1w_MNI_norm.nii.gz").exists():
        missing.append(f"T1 missing: {k}")
    if not (SRC_MSK / f"{k}_lesion_mask_MNI_clean.nii.gz").exists():
        missing.append(f"Mask missing: {k}")
if missing:
    print("WARN: some files missing:\n  " + "\n  ".join(missing[:10]) + ("" if len(missing)<=10 else f"\n  ... {len(missing)-10} more"))

# ====== Dest folders ======
OUT_TINY_T1   = STD_ROOT / "_tiny"   / "t1_norm"
OUT_TINY_MSK  = STD_ROOT / "_tiny"   / "masks_clean"
OUT_NORM_T1   = STD_ROOT / "_normal" / "t1_norm"
OUT_NORM_MSK  = STD_ROOT / "_normal" / "masks_clean"

# ====== Populate ======
def place(keys, out_t1, out_msk):
    placed = 0
    for k in keys:
        t1  = SRC_T1  / f"{k}_T1w_MNI_norm.nii.gz"
        msk = SRC_MSK / f"{k}_lesion_mask_MNI_clean.nii.gz"
        if t1.exists():
            _link_or_copy(t1,  out_t1  / t1.name)
        if msk.exists():
            _link_or_copy(msk, out_msk / msk.name)
        placed += 1
    return placed

n_tiny   = place(tiny_keys,  OUT_TINY_T1,  OUT_TINY_MSK)
n_normal = place(normal_keys, OUT_NORM_T1, OUT_NORM_MSK)

# ====== Report ======
print(f"Standardized root: {STD_ROOT}")
print(f"Total keys: {len(all_keys)} | tiny: {len(tiny_keys)} | normal: {len(normal_keys)}")
print(f"Placed tiny   → {OUT_TINY_T1.parent} : {n_tiny} cases")
print(f"Placed normal → {OUT_NORM_T1.parent} : {n_normal} cases")

# Show a couple examples
print("\nExamples (tiny):", list(sorted(tiny_keys))[:5])
print("Examples (normal):", list(sorted(normal_keys))[:5])


Standardized root: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized
Total keys: 203 | tiny: 6 | normal: 197
Placed tiny   → /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_tiny : 6 cases
Placed normal → /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_normal : 197 cases

Examples (tiny): ['sub-M2141_ses-2539', 'sub-M2145_ses-1616', 'sub-M2149_ses-677', 'sub-M2150_ses-2081', 'sub-M2201_ses-294']
Examples (normal): ['sub-M2012_ses-1158', 'sub-M2018_ses-341', 'sub-M2026_ses-3098', 'sub-M2029_ses-180', 'sub-M2034_ses-1568']


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from math import prod

# ===== ARC standardized root =====
STD = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized")
qc  = pd.read_csv(STD / "preprocess_qc.csv")

def q(a, p): return float(np.percentile(a, p)) if len(a) else np.nan

# Shapes & voxel mm³
shapes = qc[["t1_shape_x","t1_shape_y","t1_shape_z"]].dropna().astype(int).to_numpy()
voxel = qc["voxel_mm3"].dropna().to_numpy()
fov_ml = np.array([prod(s)*voxel[i]/1000.0 for i,s in enumerate(shapes)]) if len(shapes) else np.array([])

# Brain fraction
bf = qc["brain_frac"].dropna().to_numpy()

# Lesion volumes (raw & clean) in mL
lv_raw   = qc["mask_ml"].fillna(0).to_numpy()
lv_clean = qc["mask_ml_clean"].fillna(0).to_numpy()

# Changes by cleaning
changed = (qc["mask_voxels"] != qc["mask_voxels_clean"]).fillna(False).to_numpy()

# Tiny / large thresholds (you can tweak)
tiny_thr_ml  = 1.0
large_thr_ml = 200.0
n_tiny  = int((lv_clean < tiny_thr_ml).sum())
n_large = int((lv_clean > large_thr_ml).sum())

# Shape summary strings
sx, sy, sz = shapes[:,0], shapes[:,1], shapes[:,2]
shape_min = f"{sx.min()}×{sy.min()}×{sz.min()}"
shape_med = f"{int(q(sx,50))}×{int(q(sy,50))}×{int(q(sz,50))}"
shape_max = f"{sx.max()}×{sy.max()}×{sz.max()}"
vx_uniq   = ", ".join(sorted({f"{v:.3f}" for v in voxel}))

print(f"Dataset: ARC")
print(f"Cases: {len(qc)}")
print(f"Voxel mm³ (unique): {vx_uniq}")
print(f"Shape (min / median / max): {shape_min} / {shape_med} / {shape_max}")

if len(fov_ml):
    print(f"FOV volume (mL): min/median/max = {fov_ml.min():.1f} / {q(fov_ml,50):.1f} / {fov_ml.max():.1f}")

if len(bf):
    print(f"Brain fraction (nonzero/total): mean±sd = {bf.mean():.3f}±{bf.std():.3f} | p10/50/90 = {q(bf,10):.3f}/{q(bf,50):.3f}/{q(bf,90):.3f}")

print(f"Lesion mL (raw):   min/median/max = {lv_raw.min():.2f} / {q(lv_raw,50):.2f} / {lv_raw.max():.2f}")
print(f"Lesion mL (clean): min/median/max = {lv_clean.min():.2f} / {q(lv_clean,50):.2f} / {lv_clean.max():.2f}")
print(f"% masks changed by cleaning: {100.0*changed.mean():.1f}%")
print(f"Tiny lesions (<{tiny_thr_ml} mL): {n_tiny} | Large lesions (>{large_thr_ml} mL): {n_large}")

# Optional: echo where data live
print("\nFolders:")
print("  T1 normalized:", STD / "t1_norm")
print("  Masks cleaned:", STD / "masks_clean")
print("  Splits:       ", STD / "_tiny", "(tiny) |", STD / "_normal", "(normal)")


Dataset: ATLAS
Cases: 203
Voxel mm³ (unique): 1.000
Shape (min / median / max): 193×229×193 / 193×229×193 / 193×229×193
FOV volume (mL): min/median/max = 8530.0 / 8530.0 / 8530.0
Brain fraction (nonzero/total): mean±sd = 0.242±0.005 | p10/50/90 = 0.238/0.241/0.248
Lesion mL (raw):   min/median/max = 0.00 / 50.77 / 211.75
Lesion mL (clean): min/median/max = 0.00 / 50.77 / 211.75
% masks changed by cleaning: 0.5%
Tiny lesions (<1.0 mL): 6 | Large lesions (>200.0 mL): 1

Folders:
  T1 normalized: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/t1_norm
  Masks cleaned: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/masks_clean
  Splits:        /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_tiny (tiny) | /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_normal (normal)
